In [18]:
import pandas as pd

In [2]:
# Dataframes
base_path ="data/company_office/"
df_companies = pd.read_csv(base_path + 'companies.csv')
df_office_locations = pd.read_csv(base_path + 'office_locations.csv')
df_buildings = pd.read_csv(base_path + 'buildings.csv')


In [3]:
import duckdb

In [4]:
base_path ="data/company_office/"
duckdb.read_csv(base_path + 'companies.csv')
duckdb.read_csv(base_path + 'office_locations.csv')
duckdb.read_csv(base_path + 'buildings.csv')



┌───────┬─────────────────────────────────────┬─────────────┬────────┬─────────┬────────────────────┐
│  id   │                name                 │    City     │ Height │ Stories │       Status       │
│ int64 │               varchar               │   varchar   │ int64  │  int64  │      varchar       │
├───────┼─────────────────────────────────────┼─────────────┼────────┼─────────┼────────────────────┤
│     1 │ Torre KOI                           │ Monterrey   │    220 │      67 │ under construction │
│     2 │ Torre Mitikah                       │ Mexico City │    210 │      60 │ under construction │
│     3 │ Punto Chapultepec                   │ Mexico City │    210 │      59 │ proposed           │
│     4 │ Torre Reforma                       │ Mexico City │    330 │      57 │ under construction │
│     5 │ Corporativo BBVA Bancomer           │ Mexico City │    220 │      50 │ under construction │
│     6 │ Reforma 432                         │ Mexico City │    300 │     100 │ u

In [5]:
try:
    duckdb.sql("CREATE TABLE companies AS SELECT * FROM 'data/company_office/companies.csv';")
except:
    print("Table 'companies' already exists. Skipping creation.")
    pass

try:
    duckdb.sql("CREATE TABLE office_locations AS SELECT * FROM 'data/company_office/office_locations.csv';")
except:
    print("Table 'office_locations' already exists. Skipping creation.")
    pass

try:
    duckdb.sql("CREATE TABLE buildings AS SELECT * FROM 'data/company_office/buildings.csv';")
except:
    print("Table 'buildings' already exists. Skipping creation.")
    pass

In [6]:
schema_info = duckdb.sql("PRAGMA table_info('companies');").df().to_string(index=False)


In [ ]:
import ollama

# Extract the schema of the DuckDB database
def obter_schema_completo():
    # Search for all tables in the DuckDB database
    tabelas = duckdb.sql("SHOW TABLES;").df()['name'].tolist()
    
    schema_texto = "Tens acesso às seguintes tabelas e colunas no DuckDB:\n"
    
    # For each table, get its columns and types
    for tabela in tabelas:
        schema_texto += f"\n- Tabela: '{tabela}'\n  Colunas:\n"
        # Use PRAGMA table_info to get column information
        info_colunas = duckdb.sql(f"PRAGMA table_info('{tabela}');").df()
        
        for _, row in info_colunas.iterrows():
            schema_texto += f"    * {row['name']} ({row['type']})\n"
            
    return schema_texto

# Generate the schema context for the DuckDB database
contexto_tabelas = obter_schema_completo()

# Agent Rykes
system_prompt = """
Tu és um agente automatizado exclusivo para DuckDB. A tua única função é gerar código SQL.
REGRAS ESTRITAS:
1. Responde APENAS com o bloco de código SQL em formato Markdown (```sql ... ```).
2. Não incluas NENHUM texto explicativo, saudação ou notas de rodapé.
3. Usa a sintaxe nativa do DuckDB sempre que apropriado.
"""

# Input do utilizador Example
pergunta_utilizador = "Quero ver o nome dos edifícios e o status que estão na cidade de 'Mexico City' e o ano que foram movido."

# Merging the schema context with the user's question
prompt_final = f"{contexto_tabelas}\nPedido do Utilizador: {pergunta_utilizador}"

# 4. Calling the Ollama API to generate SQL code based on the system prompt and user input
result = ollama.chat(
    model='llama3',
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_final}
    ]
)

print(result['message']['content'])

```sql
SELECT b.name, b.Status, ol.move_in_year
FROM buildings b
JOIN office_locations ol ON b.id = ol.building_id
WHERE b.City = 'Mexico City'
```


In [16]:
import re

result_string = result['message']['content']

# Match ```, optional 'sql' (case-insensitive), the code, and closing ```
match = re.search(r"```(?:sql)?\s*(.*?)\s*```", result_string, re.DOTALL | re.IGNORECASE)

if match:
    clean_query = match.group(1).strip()
else:
    # Backup safety net: remove ANY leftover backticks if the block wasn't closed properly
    clean_query = re.sub(r"```(?:sql)?", "", result_string, flags=re.IGNORECASE).strip()

print("Query Limpa Pronta a Usar:")
print(clean_query)
print("-" * 40)

Query Limpa Pronta a Usar:
SELECT b.name, b.Status, ol.move_in_year
FROM buildings b
JOIN office_locations ol ON b.id = ol.building_id
WHERE b.City = 'Mexico City'
----------------------------------------


In [17]:
# Now it should run cleanly in DuckDB
query = duckdb.sql(clean_query).df().to_string(index=False)
print("Resultado da Query:")
print(query)

Resultado da Query:
                               name             Status  move_in_year
                      Torre Mitikah under construction          2022
                  Punto Chapultepec           proposed          2023
                      Torre Reforma under construction          2024
          Corporativo BBVA Bancomer under construction          2025
          Corporativo BBVA Bancomer under construction          2026
                         Reforma 90            on-hold          2027
                Torre New York Life under construction          2027
                  Punto Chapultepec           proposed          2028
                  Punto Chapultepec           proposed          2029
Residencial Vidalta Torre Altaire 3            on-hold          2031
                        Reforma 432 under construction          2022
Residencial Vidalta Torre Altaire 2            on-hold          2019
                  Punto Chapultepec           proposed          2020
              